In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("StudentsPerformance.csv")

X = df.drop(columns=["writing score", "reading score", "math score"])
y_write = df["writing score"]

categorical = X.select_dtypes(include="object").columns.tolist()

pre = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical)],
    remainder="passthrough"
)

base_pipe_write = Pipeline([
    ("pre", pre),
    ("rf", RandomForestRegressor(random_state=42, n_jobs=-1))
])

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X, y_write, test_size=0.2, random_state=42
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

best_pipe_write = None
best_params_write = None

try:
    from skopt import BayesSearchCV
    from skopt.space import Integer, Categorical
    
    search_space = {
        "rf__n_estimators": Integer(200, 1200),
        "rf__max_depth": Integer(3, 40),
        "rf__min_samples_split": Integer(2, 20),
        "rf__min_samples_leaf": Integer(1, 10),
        "rf__max_features": Categorical(["sqrt", "log2", 0.5, 0.8, 1.0]),
        "rf__bootstrap": Categorical([True, False]),
    }

    opt_write = BayesSearchCV(
        estimator=base_pipe_write,
        search_spaces=search_space,
        n_iter=30,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    opt_write.fit(X_train_w, y_train_w)
    best_pipe_write = opt_write.best_estimator_
    best_params_write = opt_write.best_params_
    print(" Bayes opt OK (writing). Best CV R2:", opt_write.best_score_)
    print("Best params:", best_params_write)

except Exception as e:
    from sklearn.model_selection import RandomizedSearchCV
    
    param_distributions = {
        "rf__n_estimators": np.arange(200, 1201, 100),
        "rf__max_depth": np.arange(3, 41, 3),
        "rf__min_samples_split": np.arange(2, 21, 2),
        "rf__min_samples_leaf": np.arange(1, 11, 1),
        "rf__max_features": ["sqrt", "log2", 0.5, 0.8, 1.0],
        "rf__bootstrap": [True, False],
    }
    
    opt_write = RandomizedSearchCV(
        estimator=base_pipe_write,
        param_distributions=param_distributions,
        n_iter=30,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    opt_write.fit(X_train_w, y_train_w)
    best_pipe_write = opt_write.best_estimator_
    best_params_write = opt_write.best_params_
    print(" Bayes opt not available -> RandomizedSearchCV used (writing). Best CV R2:", opt_write.best_score_)
    print("Best params:", best_params_write)

best_pipe_write.fit(X_train_w, y_train_w)
pred_write = best_pipe_write.predict(X_test_w)

write_metrics = {
    "r2_test": float(r2_score(y_test_w, pred_write)),
    "mae_test": float(mean_absolute_error(y_test_w, pred_write)),
    "y_test_mean": float(np.mean(y_test_w)),
}
print("\n Writing test metrics:", write_metrics)


R2 : 0.09016571738658019
MAE : 11.285045051222832
